# Chapter 08 — 작은 데이터 분석 프로젝트 완성하기

제출용 Notebook입니다. 공식 Chapter 08 흐름에 맞춰 **질문 → 데이터 점검 → 전처리 → PK/FK 검증 → 병합 → EDA → 총합 검증 → 시각화 → 해석 → 재현 확인** 순서로 작성합니다.

- 작업 위치: `assignments/chapter08/chapter08.ipynb`
- 이미지 위치: `assignments/chapter08/images/`
- 금액성 분석 기본 범위: `order_status == "completed"`


## 1. 프로젝트 질문

### 질문 1
- 카테고리별 completed 주문 기준 금액은 어떻게 다른가?

### 질문 2
- 월별 completed 주문 기준 금액과 주문 수는 어떻게 변하는가?

### 질문 3
- completed 주문 기준 구매 금액이 높은 고객은 어떤 특징을 보이는가?

### 완료 기준
- 입력 데이터 구조를 확인한다.
- PK 결측/중복과 FK 미매칭을 확인한다.
- 병합 전후 행 수와 미매칭을 확인한다.
- completed source total과 category/month/customer total이 일치하는지 확인한다.
- 대표 그래프와 해석을 남긴다.
- 전체 프로젝트 재실행 후 Validation PASS를 확인한다.


## 2. 환경 및 프로젝트 경로 확인


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
IMAGE_DIR = PROJECT_ROOT / 'assignments' / 'chapter08' / 'images'

for path in [PROCESSED_DIR, REPORT_DIR, IMAGE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python:', sys.executable)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_DIR:', RAW_DIR)
print('IMAGE_DIR:', IMAGE_DIR)


## 3. 프로젝트 함수 불러오기


In [ ]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)
from src.midterm_project import (
    build_analysis_tables,
    build_key_duplicate_checks,
    build_project_validation,
    summarize_datasets,
)


## 4. 입력 데이터 검증

사용 파일: `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv`


In [ ]:
raw_data = load_sales_data(RAW_DIR)
dataset_summary = summarize_datasets(raw_data)
display(dataset_summary)

for name, df in raw_data.items():
    print(f'\n===== {name} =====')
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    print('missing:', int(df.isna().sum().sum()))
    print('duplicates:', int(df.duplicated().sum()))


## 5. 전처리 및 PK/FK 검증


In [ ]:
processed_data = preprocess_sales_data(raw_data)
save_processed_data(processed_data, PROCESSED_DIR)
display(compare_shapes(raw_data, processed_data))

key_checks = build_key_duplicate_checks(processed_data)
relationship_checks = validate_relationships(processed_data).copy()
display(key_checks)
display(relationship_checks)


## 6. 병합·계산·EDA


In [ ]:
analysis_tables = build_analysis_tables(processed_data)

display(analysis_tables['merge_checks'])
display(analysis_tables['line_total_check'])
display(analysis_tables['date_checks'])
display(analysis_tables['amount_scope_summary'])

category_sales = analysis_tables['category_sales']
monthly_sales = analysis_tables['monthly_sales']
customer_sales_public = analysis_tables['customer_sales_public']
order_status_summary = analysis_tables['order_status_summary']

display(category_sales)
display(monthly_sales)
display(customer_sales_public.head(10))
display(order_status_summary)


## 7. Total consistency 검증


In [ ]:
total_consistency = analysis_tables['total_consistency_check']
display(total_consistency)

assert total_consistency['matches_completed'].all(), 'Total consistency Gate 실패'
print('Total consistency Gate: PASS')


## 8. 대표 시각화

그래프는 검증된 집계표를 기준으로 작성하고 `assignments/chapter08/images/`에 저장합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(category_sales['category'], category_sales['total_sales'])
ax.set_title('카테고리별 completed 주문 기준 금액')
ax.set_xlabel('카테고리')
ax.set_ylabel('금액')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(IMAGE_DIR / 'graph01.png', bbox_inches='tight')
plt.show()


## 9. 결과 해석

### 결과 관찰
- 실행 결과를 보고 실제로 확인한 사실을 작성한다.

### 나의 해석과 판단
- 왜 이 결과를 중요하게 보았는지 작성한다.

### 업무·분석적 의미
- 어떤 추가 판단이나 분석으로 연결할 수 있는지 작성한다.

### 한계와 추가 확인 사항
- 현재 데이터만으로 단정할 수 없는 내용을 작성한다.


## 10. LLM 활용 기록

- 사용 여부: 예
- 사용 목적: Notebook 구조 정리, 코드 검토, 결과 해석 보조
- Prompt 요약: Chapter 08 요구사항에 맞게 분석 흐름과 검증 항목을 확인해 달라고 요청
- 제안 요약: 실행 결과 확인 후 작성
- 반영/수정/보류: 실행 결과 확인 후 작성
- 사람이 검증한 근거: 실제 Notebook 출력과 Validation 결과를 기준으로 확인


## 11. 프로젝트 재현 확인

프로젝트 루트 터미널에서 아래 명령을 실행하고 결과를 기록합니다.

```bash
python scripts/run_midterm_project.py
```

- 재실행 결과: 실행 후 작성
- 최종 Validation 결과: 실행 후 작성
- Notebook과 핵심 수치 일치 여부: 실행 후 작성


## 12. 최종 프로젝트 요약

### 핵심 인사이트 1
- 실행 후 작성

### 핵심 인사이트 2
- 실행 후 작성

### 핵심 인사이트 3
- 실행 후 작성

### 다음 분석 제안
- 실행 후 작성
